In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import WCS
import healpy as hp
from reproject import reproject_from_healpix, reproject_to_healpix
import matplotlib.gridspec as gridspec
import matplotlib.cm as cm

from RMtools_1D import do_RMsynth_1D
from RMtools_1D import do_RMclean_1D

In [ ]:
def hpx_hdr(hdr_hpx_in, hdr_car, nside, bunit=None):

    hpx_hdr = hdr_hpx_in.copy()
    hpx_hdr['NAXIS1'] = hp.nside2npix(nside)
    hpx_hdr['NAXIS2'] = hdr_car['NAXIS3']
    hpx_hdr['NSIDE']  = nside
    hpx_hdr['CRPIX2'] = hdr_car['CRPIX3']
    hpx_hdr['CDELT2'] = hdr_car['CDELT3']
    hpx_hdr['CRVAL2'] = hdr_car['CRVAL3']
    if bunit != None:
        hpx_hdr['BUNIT']  = bunit
    try:
        del hpx_hdr['DATE']
        del hpx_hdr['INSTRUME']
        del hpx_hdr['STOKES']
    except:
        pass
    
    return hpx_hdr

## Read in OG DRAGONS (already in hpx)

In [ ]:
dragons_hdu_og     = fits.open('/srv/data/dragons/dragons-canfar-Nov2025/faraday_synthesis/dragons_FDF_clean_tot_Kgal.hpx.fits')
dragons_FD_hpx_og  = dragons_hdu_og[0].data
dragons_hdr_hpx_og = dragons_hdu_og[0].header


## Convert new DRAGONS to hpx

In [ ]:
dragons_hpx_made = False

if dragons_hpx_made:
    dragons_hdu_ch = fits.open('/srv/data/dragons/chime_compare_400_729/faraday_synthesis/FDF_clean_tot.hpx128.fits')
    dragons_FD_hpx_ch  = dragons_hdu_ch[0].data
    dragons_hdr_hpx_ch = dragons_hdu_ch[0].header
else:
    nside = dragons_hdr_hpx_og['NSIDE']
    dragons_hdu_ch = fits.open('/srv/data/dragons/chime_compare_400_729/faraday_synthesis/FDF_clean_tot.fits')
    dragons_FD_ch  = dragons_hdu_ch[0].data
    dragons_hdr_ch = dragons_hdu_ch[0].header
    print(dragons_FD_ch.shape)
    
    dragons_hdr_2D = WCS(dragons_hdr_ch).dropaxis(2).to_header()
    print(repr(dragons_hdr_2D))

    dragons_FD_hpx_ch = np.empty([dragons_FD_ch.shape[0],hp.nside2npix(nside)])
    print(dragons_FD_hpx_ch.shape)

    for i in range(0,dragons_FD_hpx_ch.shape[0]):
        print(i)
        dragons_FD_hpx_ch[i], footprint = reproject_to_healpix((dragons_FD_ch[i], dragons_hdr_2D),
                                                                coord_system_out='G',nside=nside)
        
    dragons_hdr_hpx_ch = hpx_hdr(dragons_hdr_hpx_og, dragons_hdr_ch, dragons_hdr_hpx_og['NSIDE'], bunit=None)
    
    fits.writeto('/srv/data/dragons/chime_compare_400_729/faraday_synthesis/FDF_clean_tot.hpx128.fits', 
                 dragons_FD_hpx_ch, dragons_hdr_hpx_ch, overwrite=False)


## Read in CHIME (already made it in hpx earlier)

In [ ]:
chime_hdu = fits.open('/srv/data/chime/chime_FD_May2024_400_729/FDF_clean_tot.hpx512.fits')
chime_cube_hpx = chime_hdu[0].data
chime_hdr_hpx  = chime_hdu[0].header

## Get CHIME and DRAGONS nside

In [ ]:
nside_dragons = dragons_hdr_hpx_ch['NSIDE']
nside_chime   = chime_hdr_hpx['NSIDE']

print(nside_dragons)
print(nside_chime)

## Read in convolved CHIME Q and U data

In [ ]:
chime_Q_hdu = fits.open('/srv/data/chime/chime_IQUV_Mar2024_400_729/Q_400_729_Mar2026_gal_conv.fits')
chime_U_hdu = fits.open('/srv/data/chime/chime_IQUV_Mar2024_400_729/U_400_729_Mar2026_gal_conv.fits')

chime_Q_hpx = hp.ud = chime_Q_hdu[0].data
chime_U_hpx = chime_U_hdu[0].data

freq_chime = WCS(chime_Q_hdu[0].header).all_pix2world(0,range(chime_Q_hpx.shape[0]),0)[1]
print(len(freq_chime))

print(chime_Q_hpx.shape)
print(chime_U_hpx.shape)

## Downsample CHIME FD cube to nside=256, DRAGONS FD cubes and CHIME Q, U cubes to nside=64

In [ ]:
######################
nside_coarse = 64
nside_chime_new = 256
######################

chime_FD_hpx = hp.ud_grade(chime_cube_hpx, nside_chime_new)
print(chime_FD_hpx.shape)

chime_Q_hpx = hp.ud_grade(chime_Q_hpx, nside_coarse)
print(chime_Q_hpx.shape)

chime_U_hpx = hp.ud_grade(chime_U_hpx, nside_coarse)
print(chime_U_hpx.shape)

dragons_FD_hpx_og = hp.ud_grade(dragons_FD_hpx_og, nside_coarse)
print(dragons_FD_hpx_og.shape)

dragons_FD_hpx_ch = hp.ud_grade(dragons_FD_hpx_ch, nside_coarse)
print(dragons_FD_hpx_ch.shape)

In [ ]:
def get_matched_pixels(lon, lat, nside_fine, nside_coarse):

    idx_coarse = hp.ang2pix(nside_coarse, lon, lat, lonlat=True)

    ratio = (nside_fine // nside_coarse) ** 2  # number of fine pixels per coarse pixel

    ipix_coarse_nested = hp.ring2nest(nside_coarse, idx_coarse)

    # Get fine children (NESTED), then convert back to RING
    fine_nested = np.arange(ipix_coarse_nested * ratio, (ipix_coarse_nested + 1) * ratio)
    fine_ring   = hp.nest2ring(nside_fine, fine_nested)

    #test = hp.pix2ang(nside_coarse, idx_coarse, lonlat=True)
    lon_fine,   lat_fine   = hp.pix2ang(nside_fine, fine_ring, lonlat=True)  
    lon_coarse, lat_coarse = hp.pix2ang(nside_coarse, idx_coarse, lonlat=True)  

    fig,ax = plt.subplots(1,1,figsize=(4,4))
    ax.scatter(lon_coarse, lat_coarse)
    ax.scatter(lon_fine, lat_fine)
    ax.set_aspect('equal')

    return fine_ring, idx_coarse, lon_fine, lat_fine, lon_coarse, lat_coarse

In [ ]:
lon_list = [60, 170, 90, 90,130,125,120,90,137,180,180, 30, 30, 140, ]
lat_list = [40, -35,-20,-30, 45, 45, 45,35,  7,15, -15, 60, 70, 40, ]
################
lon_t = lon_list[13]
lat_t = lat_list[13]
################

idx_chime, idx_coarse, lon_chime, lat_chime, lon_coarse, lat_coarse = get_matched_pixels(lon_t, lat_t, nside_chime_new, nside_coarse)

str_lon = str(np.round(lon_coarse,1))
str_lat = str(np.round(lat_coarse,1))

hp.mollview(chime_FD_hpx[400],min=0,max=1, title=r'$\ell=$'+str_lon+', $b=$'+str_lat)
hp.projscatter(lon_chime, lat_chime, lonlat=True, marker='o', color='white', s=1)
hp.projscatter(lon_coarse, lat_coarse, lonlat=True, marker='o', color='red', s=1)

hp.mollview(chime_Q_hpx[1],min=-1,max=1, title=r'$\ell=$'+str_lon+', $b=$'+str_lat)
hp.projscatter(lon_chime, lat_chime, lonlat=True, marker='o', color='white', s=1)
hp.projscatter(lon_coarse, lat_coarse, lonlat=True, marker='o', color='red', s=1)

In [ ]:
dq = np.array([0.01]*len(freq_chime))
du = np.array([0.01]*len(freq_chime))

chime_Q_hpx_LOS = chime_Q_hpx[:,idx_coarse]
chime_U_hpx_LOS = chime_U_hpx[:,idx_coarse]

idx_bad = np.where(abs(chime_Q_hpx_LOS) > 200)
chime_Q_hpx_LOS[idx_bad] = np.nan

idx_bad = np.where(abs(chime_U_hpx_LOS) > 200)
chime_U_hpx_LOS[idx_bad] = np.nan

mDict, aDict = do_RMsynth_1D.run_rmsynth([freq_chime, 
                                          chime_Q_hpx_LOS, 
                                          chime_U_hpx_LOS, 
                                          dq, du],
                                          phiMax_radm2=200,
                                          dPhi_radm2=0.5)

mDict_cl, aDict_cl = do_RMclean_1D.run_rmclean(mDict, aDict, 0.4)

In [ ]:
FD_ax =  np.arange(-200, 200.5, 0.5) # same FD gridding for both
cmap = cm.get_cmap('viridis') 
colors = cmap(np.linspace(0, 1, len(idx_chime)))
fs = 16

fig, ax = plt.subplots(1,1, figsize=(16,6))
ax2 = ax.twinx()
#for i in range(0,len(idx_chime)):
#    ax2.plot(FD_ax, chime_FD_hpx[:,idx_chime[i]], color=colors[i], linewidth=1)

ax.plot(FD_ax, dragons_FD_hpx_og[:,idx_coarse], color='k', linewidth=3, label='DRAGONS 350-1030 MHz')
ax.plot(FD_ax, dragons_FD_hpx_ch[:,idx_coarse], color='grey', linewidth=3, label='DRAGONS 400-729 MHz')
ax2.plot(aDict_cl['phiArr_radm2'], abs(aDict_cl['cleanFDF']), color='blue', linewidth=3, label='low-res CHIME 400-729 MHz')
ax2.plot(FD_ax, np.nanmean(chime_FD_hpx[:,idx_chime],axis=1), color='red', linewidth=3, label='mean PI CHIME 400-729 MHz')


ax.set_xlim(-50,50)
ax.set_ylim(0,  1.05*np.nanmax(dragons_FD_hpx_og[:,idx_coarse]))
ax2.set_ylim(0, 1.05*np.nanmax(np.nanmean(chime_FD_hpx[:,idx_chime],axis=1)))

ax.set_xticks(np.arange(-50,55,5))
ax.grid(axis='x')

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(2)

ax.tick_params(axis='both', labelsize=fs, left=True, right=False, 
                   top=True, which='both', width=2, length=6)

ax2.tick_params(axis='both', labelsize=fs, left=False, right=True, 
                   top=True, which='both', width=2, length=6)

#ax.set_zorder(ax2.get_zorder()+1)
#ax.patch.set_visible(False)

ax2.set_ylabel('CHIME PI (Jy/beam/RMSF)',fontsize=fs)
ax.set_ylabel('DRAGONS PI (K/RMSF)',fontsize=fs)
ax.set_xlabel(r'Faraday depth (rad/m$^2$)',fontsize=fs)

ax.legend(fontsize=fs, loc='upper left', framealpha=1)
ax2.legend(fontsize=fs, loc='upper right', framealpha=1)

fig.suptitle(r'$\ell=$'+str_lon+', $b=$'+str_lat, fontsize=fs+4)

plt.savefig(f'/home/aordog/Dropbox/plots/gmims/chime-dragons/spectra_May2026/spectra_{str_lon}_{str_lat}.png')